[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aditya-siraskar/PINN_FTRE/blob/main/notebooks/02_phase1_stage2b_pinn1d.ipynb)

# 02 · Stage 2b: 1-D PINN for the jump-diffusion PIDE

**Phase:** Phase 1: validation (1-D)

**Goal:** Train the tail-focused PINN and measure density accuracy (bulk vs tail, RQ1) and VaR/ES accuracy against the exact answer.

**Outputs:** `figures/stage2_pinn1d/fig01–fig05`, `results/stage2_pinn1d/{metrics.json, history.csv, risk_table.csv, model.pt}`

> Runtime → Change runtime type → **T4 GPU**. Run the setup cell first. With `USE_DRIVE = True` the repo lives in
> `MyDrive/PINN_FTRE`, so figures and results persist. Core logic lives in `pinn_tailrisk/` and `scripts/`;
> this notebook only runs it and shows the results.

In [ ]:
# @title Setup: repo on Google Drive, install, GPU check  { display-mode: "form" }
USE_DRIVE = True   # keep figures/results across Colab sessions (recommended)
REPO_URL = "https://github.com/aditya-siraskar/PINN_FTRE"

import os, sys
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive"
else:
    BASE = "/content"
ROOT = f"{BASE}/PINN_FTRE"
if not os.path.exists(ROOT):
    !git clone -q {REPO_URL} "{ROOT}"
else:
    !git -C "{ROOT}" pull -q --ff-only
%cd "{ROOT}"
!pip install -q -e .
sys.path.insert(0, ROOT)

import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (Runtime > Change runtime type > T4 GPU)")

In [ ]:
# autoreload picks up library edits without a restart. Colab's IPython build imports the
# removed `imp` module on Python >= 3.12, so it is optional: if it is skipped, use
# Runtime > Restart session after editing pinn_tailrisk/.
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except ModuleNotFoundError:
    print("autoreload unavailable here; restart the session after editing the library")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pinn_tailrisk import plotting as P
from pinn_tailrisk.notebook import show_figures, load_metrics
P.use_style()

## 1. Hyperparameters (`configs/stage2_pinn1d.yaml`)

In [ ]:
print(open('configs/stage2_pinn1d.yaml').read())

## 2. Smoke run
500 steps: checks that the code runs and measures speed. Results go to `stage2_pinn1d_smoke`.

In [ ]:
!python scripts/stage2_pinn1d.py --steps 500 --tag smoke

In [ ]:
h = pd.read_csv("results/stage2_pinn1d_smoke/history.csv")
print(f"{h.step.iloc[-1] / h.time.iloc[-1]:.1f} steps/s  ->  full run of 20k steps ≈ {20000 / (h.step.iloc[-1] / h.time.iloc[-1]) / 60:.0f} min")

## 3. Full training run

In [ ]:
!python scripts/stage2_pinn1d.py

## 4. Figures and what to check

| Figure | Pass criterion |
|---|---|
| fig01_training | all loss terms decrease; ES₉₉ error ends below 1% at 1 d and 10 d |
| fig02_density | PINN on top of the exact curve inside the worst-1% band |
| fig03_error_map | no systematic error band near the 1% quantile line |
| fig04_bulk_vs_tail | **RQ1**: how much worse the tail is than the bulk, for PINN vs FD |
| fig05_var_es_error | PINN VaR/ES errors at every α, compared with FD |

In [ ]:
show_figures("stage2_pinn1d")

## 5. Numbers

In [ ]:
m = load_metrics("stage2_pinn1d")
print(f"train {m['train_time_s']/60:.1f} min | PINN VaR/ES query {m['pinn_query_ms']:.1f} ms | FD solve {m['fd_solve_s']*1e3:.0f} ms")
display(pd.DataFrame(m["density_error"]).T.style.format("{:.2e}"))
r = pd.read_csv("results/stage2_pinn1d/risk_table.csv")
display(r[["h", "alpha", "var_exact", "var_pinn", "es_exact", "es_pinn", "var_rel_pinn", "es_rel_pinn", "es_rel_fd"]].style.format(precision=5))